In [1]:
!pip -q install langchain-groq
!pip -q install -U langchain_community tiktoken langchainhub
!pip -q install -U langchain langgraph
!pip -q install -U langchain langchain-community langchainhub
!pip -q install langchain-chroma bs4
!pip -q install huggingface_hub unstructured sentence_transformers
!pip -q install langchain_huggingface

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()

groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv("HF_TOKEN")

os.environ["GROQ_API_KEY"] = groq_api_key
os.environ["HF_TOKEN"] = hf_token

In [3]:
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(model_name="microsoft/graphcodebert-base")

/workspaces/BASH-AGENT-LANGGRAPH/.conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
No sentence-transformers model found with name microsoft/graphcodebert-base. Creating a new one with mean pooling.
Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
import json
from langchain.docstore.document import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma


json_filepath = './bash_commands_full.json'


with open(json_filepath, 'r', encoding='utf-8') as f:
    command_data = json.load(f)


docs = []
for item in command_data:
    docs.append(Document(
        page_content=item['description'],
        metadata={'command': item['command']}
    ))


docs = [doc for doc in docs if 'file' in doc.page_content.lower().split()]


splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)
chunked_docs = splitter.split_documents(docs)


chroma_db = Chroma.from_documents(
    documents=chunked_docs,
    collection_name='rag_linux_commands',
    embedding=embeddings,
    collection_metadata={"hnsw:space": "cosine"},
    persist_directory="./linux_cmd_db"
)

chunked_docs[:3]


[Document(metadata={'command': 'addr2line'}, page_content='Used to convert addresses into file names and line numbers.'),
 Document(metadata={'command': 'autoupdate'}, page_content='Update a configure.in file to newer autoconf.'),
 Document(metadata={'command': 'bzip2'}, page_content='A block-sorting file compressor used to shrink given files.')]

In [5]:
retriever = chroma_db.as_retriever(search_kwargs={"k": 5})

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough

from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

In [7]:
from langchain_groq import ChatGroq

GROQ_LLM = ChatGroq(
            model="llama3-70b-8192",
        )

In [8]:

rag_prompt = PromptTemplate(
    template="""
You are an assistant for answering questions about Bash commands.
Use the provided context from command descriptions to answer the question.
If you don’t know the answer, just say “I don’t know.”
Return just your code no extra word just the code.

QUESTION: {question}

CONTEXT:
{context}

Answer:
""",
    input_variables=["question", "context"],
)

# Build the chain
rag_prompt_chain = rag_prompt | GROQ_LLM | StrOutputParser()

# Example usage
QUESTION = "Used to convert addresses into file names and line numbers"
CONTEXT = retriever.invoke(QUESTION)


result = rag_prompt_chain.invoke({"question": QUESTION, "context": CONTEXT})

print("Answer:", result)

Answer: addr2line


In [9]:
rag_chain = (
    {"context": retriever , "question": RunnablePassthrough()}
    | rag_prompt
    | GROQ_LLM
    | StrOutputParser()
)

In [10]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.prompts import PromptTemplate

from langchain_core.output_parsers import StrOutputParser
from langchain_core.output_parsers import JsonOutputParser

# UTILS

In [17]:

def write_markdown_file(content, filename):
    """Writes the given content as a markdown file to the local directory.

    Args:
        content: The content to write to the file (string, list, dict, or StringPromptValue).
        filename: The filename to save the file as (without extension).
    """
    # Convert LangChain's StringPromptValue (or similar) to plain string
    if hasattr(content, "to_string"):
        content = content.to_string()
    # Fallback for any non-str object: use str()
    elif not isinstance(content, str):
        if isinstance(content, dict):
            # Render dict as key: value lines
            content = "\n".join(f"{k}: {v}" for k, v in content.items())
        elif isinstance(content, list):
            # Render list of strings as newline-separated
            content = "\n".join(str(item) for item in content)
        else:
            # Generic fallback
            content = str(content)

    # Write out the markdown file
    with open(f"{filename}.md", "w", encoding="utf-8") as f:
        f.write(content)



In [18]:
bash_code_prompt = PromptTemplate(
    template="""
You are a Bash scripting expert. 
When given a description of a task, output only the Bash code (no commentary) that accomplishes it.
Make the script robust: include comments, error-checking where appropriate, and use best practices.

TASK DESCRIPTION:
{initial_prompt}

# YOUR BASH SCRIPT:
""",
    input_variables=["initial_prompt"],
)

# Build the chain
bash_code_generator = bash_code_prompt | GROQ_LLM | StrOutputParser()

# Example usage
TASK = "List all files in the current directory (including hidden), sort them by modification time descending, and save the output to a file named files.txt. Exit with error if the directory is not accessible."

script = bash_code_generator.invoke({"initial_prompt": TASK})
print(script)


```bash
#!/bin/bash

# Check if current directory is accessible
if [ ! -r "." ]; then
  echo "Error: Cannot access current directory." >&2
  exit 1
fi

# List all files in the current directory, sort by modification time descending, and save to files.txt
ls -aqt | sort -r > files.txt
```


In [19]:
# Research Router Prompt — routes to either code generation or clarification
research_router_prompt = PromptTemplate(
    template="""
You are a Bash assistant router: you read a user’s task description and task category,
and decide whether to generate ready-to-use Bash code (`generate_code`) or
ask the user for more details (`request_clarification`).

Routing rules:
- If the task is clear, self-contained, and you know how to implement it in Bash → `generate_code`.
- If the task is ambiguous, missing critical details, or requires user-specific context → `request_clarification`.

Return ONLY a JSON object with a single key `router_decision` whose value is either `generate_code` or `request_clarification`. No extra text.

TASK DESCRIPTION:
{initial_prompt}

TASK CATEGORY:
{prompt_category}
""",
    input_variables=["initial_email", "email_category"],
)

research_router = research_router_prompt | GROQ_LLM | JsonOutputParser()

# Example
Prompt = "I need to recursively find all `.log` files modified in the last 7 days and compress them into a tar.gz. How do I do that?"
Prompt_category = "command_enquiry"

decision = research_router.invoke({
    "initial_prompt": Prompt,
    "prompt_category": Prompt_category
})

print(decision)  # -> {"router_decision": "generate_code"} or {"router_decision": "request_clarification"}


{'router_decision': 'generate_code'}


In [20]:
# RAG Questions Prompt — generates clarifying/internal questions
search_rag_prompt = PromptTemplate(
    template="""
You are an expert at designing the best internal queries to retrieve precise information for Bash scripting tasks.

Given the TASK DESCRIPTION and TASK CATEGORY, produce up to three concise questions to ask your internal Bash knowledge system. These questions should surface details needed to write a robust, correct script.

Return ONLY a JSON object with key "questions" whose value is a list of strings (no more than 3), and no additional text.

TASK DESCRIPTION:
{initial_prompt}

TASK CATEGORY:
{prompt_category}
""",
    input_variables=["initial_prompt", "prompt_category"],
)

question_rag_chain = search_rag_prompt | GROQ_LLM | JsonOutputParser()

# Example
Prompt = "I need to back up all `.conf` files from /etc recursively, but exclude any files larger than 1MB."
prompt_category = "command_enquiry"

result = question_rag_chain.invoke({
    "initial_prompt": Prompt,
    "prompt_category": prompt_category
})

print(result)  # -> {"questions": ["Should the backup preserve directory structure?", "Do we include hidden .conf files?", "..."]}


{'questions': ['How can I use `find` to recursively search for files with a specific extension?', 'What option can I use with `find` to exclude files larger than a certain size?', 'How can I use `xargs` or another command to copy the found files to a backup location?']}


In [21]:



draft_writer_prompt = PromptTemplate(
    template="""
You are a Bash scripting assistant. Based on the user’s TASK_DESCRIPTION, TASK_CATEGORY, and provided RESEARCH_INFO:

• If TASK_CATEGORY == "command_enquiry", produce a ready-to-run Bash script.  
• Otherwise, produce a clarification question.

**Output format**:
1) A JSON object with a single key:
   - `"script_md"` if you’re returning a script  
   - `"clarification"` if you’re asking for more info  
2) Immediately after the JSON, include the script as a Markdown fenced code block (```bash … ```), or just plain text for clarification.

No extra text or explanations.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}
""",
    input_variables=["initial_prompt", "prompt_category", "research_info"],
)

draft_writer_chain = draft_writer_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 30 days in /var/log and email me if any errors occur."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime +30, tar with exit-code check, then send mail via mailx."

response = draft_writer_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info
})

print(response)  # -> {"script_md": ""} followed by the fenced code block


OutputParserException: Invalid json output: {"script_md"}

```bash
#!/bin/bash

# Archive all .log files older than 30 days in /var/log
find /var/log -name "*.log" -mtime +30 -print0 | tar --null -T - -czf log_archive.tar.gz

# Check if tar command was successful
if [ $? -ne 0 ]; then
  echo "Error archiving log files" | mailx -s "Error archiving log files" your_email@example.com
  exit 1
fi
```
For troubleshooting, visit: https://python.langchain.com/docs/troubleshooting/errors/OUTPUT_PARSING_FAILURE 

In [ ]:
# Rewrite Router Prompt — decides whether the generated script/clarification is sufficient
rewrite_router_prompt = PromptTemplate(
    template="""
You are an expert at evaluating Bash scripts and clarification questions that have been generated for a user task.
Compare the original TASK_DESCRIPTION and TASK_CATEGORY to the generated OUTPUT (either a script or clarification).
Decide if the OUTPUT fully addresses the user’s needs:

- If OUTPUT is a full script and it implements all requirements from the TASK_DESCRIPTION → 'no_rewrite'
- If OUTPUT is a clarification question and it asks for missing details → 'no_rewrite'
- Otherwise → 'rewrite'

Return ONLY a JSON object with key 'router_decision' whose value is either 'rewrite' or 'no_rewrite'. No extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

GENERATED_OUTPUT:
{draft_code}
""",
    input_variables=["initial_prompt", "prompt_category", "draft_code"],
)

rewrite_router = rewrite_router_prompt | GROQ_LLM | JsonOutputParser()

# Example
prompt = "Archive all .log files older than 7 days under /var/log and report failures via email."
prompt_category = "command_enquiry"
draft_code = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
# Missing email notification logic
"""

decision = rewrite_router.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "draft_code": draft_code
})

print(decision)


{'router_decision': 'rewrite'}


In [ ]:
# Draft Analysis Prompt — provides feedback on the generated script or clarification
draft_analysis_prompt = PromptTemplate(
    template="""
You are the Quality Control Agent for Bash scripting tasks.
Read the original TASK_DESCRIPTION, the TASK_CATEGORY, and any RESEARCH_INFO provided.
Then analyze the GENERATED_OUTPUT (either a Bash script or a clarification question).

- If it's a script: check that it meets all requirements, follows best practices, includes error handling, and is clear.
- If it's a clarification: check it asks for the precise missing details.

Give concise, actionable feedback on what can be improved or added. Do NOT introduce any new facts.

Return ONLY a JSON object with key "draft_analysis" whose value is the feedback string. No extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}

GENERATED_OUTPUT:
{draft_code}
""",
    input_variables=["initial_prompt", "prompt_category", "research_info", "draft_code"],
)

draft_analysis_chain = draft_analysis_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 7 days under /var/log and report failures via email."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime and tar; then check exit code of tar and send mail."
draft_code = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
"""

analysis = draft_analysis_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info,
    "draft_code": draft_code
})

print(analysis)  # -> {"draft_analysis": "...actionable feedback..."}


{'draft_analysis': "The script is close, but it's missing error handling and email reporting. Consider adding a check for the exit code of tar and using a mail command to report failures. Additionally, it's a good practice to specify the full path to the tar command and to redirect the output to a log file for debugging purposes. Also, the script should handle the case when no files are found."}


In [ ]:


# Rewrite Script Prompt — improves the draft script or clarification based on QC feedback
rewrite_script_prompt = PromptTemplate(
    template="""
You are the Final Bash Script Agent. Using the QC feedback, rewrite and improve the draft Bash script or clarification
to fully meet the user’s TASK_DESCRIPTION and requirements.

Do NOT add any facts or details not present in the RESEARCH_INFO or the original TASK_DESCRIPTION.

Return ONLY a JSON object with key "final_output" whose value is the improved Bash script (or clarification question, if that was the draft). No extra text.

TASK_DESCRIPTION:
{initial_prompt}

TASK_CATEGORY:
{prompt_category}

RESEARCH_INFO:
{research_info}

DRAFT_OUTPUT:
{draft_code}

QC_FEEDBACK:
{code_analysis}
""",
    input_variables=[
        "initial_prompt",
        "prompt_category",
        "research_info",
        "draft_code",
        "code_analysis",
    ],
)

rewrite_chain = rewrite_script_prompt | GROQ_LLM | JsonOutputParser()

# Example usage
prompt = "Archive all .log files older than 7 days under /var/log and notify me if compression fails."
prompt_category = "command_enquiry"
research_info = "Use find with -mtime +7, tar with exit-code check, then send mail via mailx."
draft_script = """#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz
"""

qc_feedback = "The script compresses files but lacks error handling and email notification."

result = rewrite_chain.invoke({
    "initial_prompt": prompt,
    "prompt_category": prompt_category,
    "research_info": research_info,
    "draft_code": draft_script,
    "code_analysis": qc_feedback
})

print(result["final_output"])
# -> Improved Bash script with proper error checks and mailx notification


#!/usr/bin/env bash
find /var/log -name '*.log' -mtime +7 | xargs tar -czf logs.tar.gz || echo 'Compression failed' | mailx -s 'Compression failed' your_email@example.com


In [ ]:
from langchain.schema import Document
from langgraph.graph import END, StateGraph

In [ ]:
from typing_extensions import TypedDict
from typing import List

### State

class GraphState(TypedDict):
    """
    Represents the state of our graph.

    Attributes:
        initial_prompt: email
        prompt_category: email category
        draft_code: LLM generation
        final_code: LLM generation
        research_info: list of documents
        info_needed: whether to add search info
        num_steps: number of steps
    """
    initial_prompt : str
    prompt_category : str
    draft_code : str
    final_code : str
    research_info : List[str] # this will now be the RAG results
    info_needed : bool
    num_steps : int
    draft_code_feedback : dict
    rag_questions : List[str]

In [ ]:
def categorize_prompt(state):
    """take the initial prompt and categorize it"""
    print("---CATEGORIZING INITIAL PROMPT---")
    initial_prompt = state['initial_prompt']
    num_steps = int(state['num_steps'])
    num_steps += 1

    prompt_category = bash_code_prompt.invoke({"initial_prompt": initial_prompt})
    print(prompt_category)
    # save to local disk
    write_markdown_file(prompt_category, "prompt_category")

    return {"prompt_category": prompt_category, "num_steps":num_steps}

In [ ]:
def research_info_search(state):

    print("---RESEARCH INFO RAG---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    num_steps = state['num_steps']
    num_steps += 1

    # Web search
    questions = question_rag_chain.invoke({"initial_prompt": initial_prompt,
                                            "prompt_category": prompt_category })
    questions = questions['questions']
    # print(questions)
    rag_results = []
    for question in questions:
        print(question)
        temp_docs = rag_chain.invoke(question)
        print(temp_docs)
        question_results = question + '\n\n' + temp_docs + "\n\n\n"
        if rag_results is not None:
            rag_results.append(question_results)
        else:
            rag_results = [question_results]
    print(rag_results)
    print(type(rag_results))
    write_markdown_file(rag_results, "research_info")
    write_markdown_file(questions, "rag_questions")
    return {"research_info": rag_results,"rag_questions":questions, "num_steps":num_steps}

In [ ]:
def draft_email_writer(state):
    print("---DRAFT CODE WRITER---")
    # Get inputs from state
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    research_info = state["research_info"]
    num_steps = state["num_steps"]
    num_steps += 1

    # Invoke the draft writer chain
    draft_code = draft_writer_chain.invoke({
        "initial_prompt": initial_prompt,
        "prompt_category": prompt_category,
        "research_info": research_info
    })
    print(draft_code)

    # **Modified:** Use 'script_md' key if present (chain returns {'script_md': ...})
    code_draft = draft_code.get("script_md", draft_code.get("draft_code", ""))
    write_markdown_file(code_draft, "draft_code")

    return {"draft_code": code_draft, "num_steps": num_steps}


In [ ]:
def analyze_draft_email(state):
    print("---DRAFT CODE ANALYZER---")
    # Get the state
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]
    research_info = state["research_info"]
    num_steps = state["num_steps"]
    num_steps += 1

    # Invoke the draft analysis chain
    analysis_dict = draft_analysis_chain.invoke({
        "initial_prompt": initial_prompt,
        "prompt_category": prompt_category,
        "research_info": research_info,
        "draft_code": draft_code
    })

    # **Modified:** Extract just the feedback string
    feedback_str = analysis_dict.get("draft_analysis", "")
    write_markdown_file(feedback_str, "draft_code_feedback")

    return {"draft_code_feedback": feedback_str, "num_steps": num_steps}


In [ ]:
def rewrite_email(state):
    print("---REWRITE CODE---")
    # Get the state
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]
    research_info = state["research_info"]
    draft_code_feedback = state["draft_code_feedback"]
    num_steps = state["num_steps"]
    num_steps += 1

    # Invoke the rewrite chain
    result = rewrite_chain.invoke({
        "initial_prompt": initial_prompt,
        "prompt_category": prompt_category,
        "research_info": research_info,
        "draft_code": draft_code,
        "code_analysis": draft_code_feedback
    })

    # **Modified:** Extract the final script text from result
    final_output = result.get("final_output", result.get("final_code", ""))
    write_markdown_file(final_output, "final_code")

    return {"final_code": final_output, "num_steps": num_steps}


In [ ]:
def no_rewrite(state):
    print("---NO REWRITE CODE ---")
    ## Get the state
    draft_code = state["draft_code"]
    num_steps = state['num_steps']
    num_steps += 1

    write_markdown_file(str(draft_code), "final_code")
    return {"final_code": draft_code, "num_steps":num_steps}

In [ ]:
def state_printer(state):
    """print the state"""
    print("---STATE PRINTER---")
    print(f"Initial Prompt: {state['initial_prompt']} \n" )
    print(f"Prompt Category: {state['prompt_category']} \n")
    print(f"Draft Code: {state['draft_code']} \n" )
    print(f"Final Code: {state['final_code']} \n" )
    print(f"Research Info: {state['research_info']} \n")
    print(f"RAG Questions: {state['rag_questions']} \n")
    print(f"Num Steps: {state['num_steps']} \n")
    return

# Conditional Edges

In [ ]:
def route_to_research(state):
    """
    Route email to web search or not.
    Args:
        state (dict): The current graph state
    Returns:
        str: Next node to call
    """

    print("---ROUTE TO RESEARCH---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]


    router = research_router.invoke({"initial_prompt": initial_prompt,"prompt_category":prompt_category })
    print(router)
    print(router['router_decision'])
    if router['router_decision'] == 'research_info':
        print("---ROUTE prompt TO RESEARCH INFO---")
        return "research_info"
    elif router['router_decision'] == 'draft_code':
        print("---ROUTE prompt TO DRAFT CODE---")
        return "draft_code"

In [ ]:
def route_to_rewrite(state):

    print("---ROUTE TO REWRITE---")
    initial_prompt = state["initial_prompt"]
    prompt_category = state["prompt_category"]
    draft_code = state["draft_code"]


    router = rewrite_router.invoke({"initial_prompt": initial_prompt,
                                     "prompt_category":prompt_category,
                                     "draft_code":draft_code}
                                   )
    print(router)
    print(router['router_decision'])
    if router['router_decision'] == 'rewrite':
        print("---ROUTE TO ANALYSIS - REWRITE---")
        return "rewrite"
    elif router['router_decision'] == 'no_rewrite':
        print("---ROUTE CODE TO FINAL CODE---")
        return "no_rewrite"

In [ ]:

workflow = StateGraph(GraphState)

# Define the nodes
workflow.add_node("categorize_prompt", categorize_prompt) # categorize email
workflow.add_node("research_info_search", research_info_search) # web search
workflow.add_node("state_printer", state_printer)
workflow.add_node("draft_email_writer", draft_email_writer)
workflow.add_node("analyze_draft_email", analyze_draft_email)
workflow.add_node("rewrite_email", rewrite_email)
workflow.add_node("no_rewrite", no_rewrite)



In [ ]:
workflow.set_entry_point("categorize_prompt")


workflow.add_edge("categorize_prompt", "research_info_search")
workflow.add_edge("research_info_search", "draft_email_writer")


workflow.add_conditional_edges(
    "draft_email_writer",
    route_to_rewrite,
    {
        "rewrite": "analyze_draft_email",
        "no_rewrite": "no_rewrite",
    },
)
workflow.add_edge("analyze_draft_email", "rewrite_email")
workflow.add_edge("no_rewrite", "state_printer")
workflow.add_edge("rewrite_email", "state_printer")
workflow.add_edge("state_printer", END)

In [ ]:
# Compile
app = workflow.compile()

In [ ]:
EMAIL = "Compress regular files in the testbed directory tree that were last modified more than 7 days ago"
# run the agent
inputs = {"initial_prompt": EMAIL, "num_steps":0}
for output in app.stream(inputs):
    for key, value in output.items():
        pprint(f"Finished running: {key}:")

---CATEGORIZING INITIAL PROMPT---
text='\nYou are a Bash scripting expert. \nWhen given a description of a task, output only the Bash code (no commentary) that accomplishes it.\nMake the script robust: include comments, error-checking where appropriate, and use best practices.\n\nTASK DESCRIPTION:\nCompress regular files in the testbed directory tree that were last modified more than 7 days ago\n\n# YOUR BASH SCRIPT:\n'
'Finished running: categorize_prompt:'
---RESEARCH INFO RAG---
How can I recursively traverse the testbed directory tree?
find .
How can I identify regular files that were last modified more than 7 days ago?
find . -type f -mtime +7
What is the best way to compress these files while preserving their original path and filename?
tar -czf output.tar.gz input/*
['How can I recursively traverse the testbed directory tree?\n\nfind .\n\n\n', 'How can I identify regular files that were last modified more than 7 days ago?\n\nfind . -type f -mtime +7\n\n\n', 'What is the best way